In [31]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    precision_recall_fscore_support
)
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import sys

In [32]:
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)

Python version: 3.10.16 (main, Dec  3 2024, 17:27:57) [Clang 16.0.0 (clang-1600.0.26.4)]
PyTorch version: 2.5.1


In [33]:
def set_seed(seed=42):
    """Set random seed for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    elif torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

In [34]:
class DDosDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [35]:
class AdvancedDDosDetector(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        # More adaptive architecture
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        return self.layers(x)

In [36]:

def robust_preprocess_ddos_data(df, test_size=0.2, random_state=42, 
                                 correlation_threshold=0.5, 
                                 importance_threshold=0.05,
                                 variance_threshold=0.01):
    """
    Enhanced preprocessing with advanced feature selection and filtering
    """

    # Clean column names by stripping whitespace
    df.columns = df.columns.str.strip()
    
    # Strip whitespace from all string columns
    for col in df.select_dtypes(include=['object']):
        df[col] = df[col].str.strip()
    
    # Identify the label column
    label_col = 'Label'
    
    # Separate features and target
    y = df[label_col]
    X = df.drop(columns=[label_col])
    
    def clean_numeric_columns(df):
        # Replace infinite values with NaN
        df = df.replace([np.inf, -np.inf], np.nan)
        
        # Identify numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        # Remove low variance features
        from sklearn.feature_selection import VarianceThreshold
        selector = VarianceThreshold(threshold=variance_threshold)
        
        # Fit and transform
        selector.fit(df[numeric_cols])
        high_variance_cols = numeric_cols[selector.get_support()]
        
        # For each numeric column, clip extreme values
        for col in high_variance_cols:
            # Calculate safe bounds using percentiles
            lower = df[col].quantile(0.001)
            upper = df[col].quantile(0.999)
            
            # Clip values to these bounds
            df[col] = df[col].clip(lower=lower, upper=upper)
        
        return df[high_variance_cols]
    
    # Clean numeric columns
    X = clean_numeric_columns(X)
    
    def advanced_correlation_filtering(X, y, threshold=correlation_threshold):
        # Create a copy of the dataframe
        X_with_label = X.copy()
        
        # Encode the label column numerically
        le = LabelEncoder()
        label_encoded = le.fit_transform(y)
        
        # Add encoded label to the dataframe
        X_with_label['Label'] = label_encoded
        
        # Compute correlation with the label column
        label_correlations = X_with_label.corr()['Label'].abs()
        
        # Remove the label from correlation list to avoid comparing label with itself
        label_correlations = label_correlations.drop('Label')
        
        # Select features that have correlation above the threshold with the label
        selected_features = label_correlations[label_correlations > threshold].index.tolist()
        
        # If no features meet the threshold, return all features
        if not selected_features:
            print("Warning: No features found above correlation threshold. Returning all features.")
            return X
        
        return X[selected_features]
    
    def advanced_feature_filtering(X, y, threshold=importance_threshold):
        # Compute feature importance using Random Forest
        rf = RandomForestClassifier(
            n_estimators=100, 
            random_state=random_state, 
            class_weight='balanced'  # Handle class imbalance
        )
        rf.fit(X, y)
        
        # Get feature importances
        importances = pd.DataFrame({
            'feature': X.columns,
            'importance': rf.feature_importances_
        })
        
        # Select features above importance threshold
        important_features = importances[
            importances['importance'] > threshold
        ]['feature'].tolist()
        
        return X[important_features]
    
    # Print initial class distribution
    print("Original Class Distribution:")
    print(y.value_counts(normalize=True))
    
    # Apply advanced feature selection techniques
    X = advanced_correlation_filtering(X, y)
    X = advanced_feature_filtering(X, y)
    
    # Handle any remaining missing values
    X.fillna(X.median(), inplace=True)
    
    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # Stratified split with shuffling
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=test_size, random_state=random_state, 
        stratify=y_encoded, shuffle=True
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Print final dataset info
    print("\nFinal Dataset Info:")
    print(f"Total features: {X_train_scaled.shape[1]}")
    print(f"Train samples: {X_train_scaled.shape[0]}")
    print(f"Test samples: {X_test_scaled.shape[0]}")
    print("\nClass Distribution:")
    print("Train:", np.unique(y_train, return_counts=True)[1] / len(y_train))
    print("Test:", np.unique(y_test, return_counts=True)[1] / len(y_test))
    
    # Visualize feature importance
    rf_classifier = RandomForestClassifier(
        n_estimators=100, 
        random_state=random_state, 
        class_weight='balanced'
    )
    rf_classifier.fit(X_train_scaled, y_train)
    
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_classifier.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(12,6))
    feature_importance.head(20).plot(x='feature', y='importance', kind='bar')
    plt.title("Top 20 Feature Importances")
    plt.xlabel("Features")
    plt.ylabel("Importance")
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.close()
    
    return {
        'X_train': X_train_scaled, 
        'X_test': X_test_scaled, 
        'y_train': y_train, 
        'y_test': y_test,
        'scaler': scaler,
        'label_encoder': le,
        'input_dim': X_train_scaled.shape[1],
        'num_classes': len(le.classes_),
        'feature_names': X.columns.tolist()
    }

In [37]:
def train_ddos_detector(model, train_loader, val_loader, criterion, optimizer, 
                        num_epochs=30, patience=7, device=None):
    """
    Enhanced training loop with advanced regularization and learning rate scheduling
    """
    # Device selection
    device = torch.device(
        'cuda' if torch.cuda.is_available() else 
        'mps' if torch.backends.mps.is_available() else 
        'cpu'
    )
    
    print(f"Using device: {device}")
    model = model.to(device)
    
    # Learning rate scheduler with cosine annealing
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=num_epochs, 
        eta_min=1e-5
    )
    
    best_val_loss = float('inf')
    patience_counter = 0
    training_history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    # Training progress
    for epoch in tqdm(range(num_epochs), desc="Training Progress"):
        model.train()
        total_train_loss = 0
        total_train_correct = 0
        total_train_samples = 0
        
        # Training phase
        for batch_features, batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)  # Gradient clipping
            optimizer.step()
            
            total_train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_train_correct += (predicted == batch_labels).sum().item()
            total_train_samples += batch_labels.size(0)
        
        # Validation phase
        model.eval()
        total_val_loss = 0
        total_val_correct = 0
        total_val_samples = 0
        
        with torch.no_grad():
            for batch_features, batch_labels in val_loader:
                batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
                
                outputs = model(batch_features)
                loss = criterion(outputs, batch_labels)
                
                total_val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_val_correct += (predicted == batch_labels).sum().item()
                total_val_samples += batch_labels.size(0)
        
        # Metrics calculation
        train_loss = total_train_loss / len(train_loader)
        train_accuracy = total_train_correct / total_train_samples
        val_loss = total_val_loss / len(val_loader)
        val_accuracy = total_val_correct / total_val_samples
        
        # Store history
        training_history['train_loss'].append(train_loss)
        training_history['train_acc'].append(train_accuracy)
        training_history['val_loss'].append(val_loss)
        training_history['val_acc'].append(val_accuracy)
        
        print(f"Epoch {epoch+1}: "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")
        
        # Early stopping and model saving
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_ddos_model.pth')
        else:
            patience_counter += 1
        
        scheduler.step()
        
        if patience_counter >= patience:
            print("Early stopping triggered")
            break
    
    # Plot training history
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(training_history['train_loss'], label='Train Loss')
    plt.plot(training_history['val_loss'], label='Val Loss')
    plt.title('Loss Curve')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(training_history['train_acc'], label='Train Accuracy')
    plt.plot(training_history['val_acc'], label='Val Accuracy')
    plt.title('Accuracy Curve')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.close()
    
    return model, training_history

In [38]:
def comprehensive_model_validation(X, y, model_class, input_dim, num_classes, n_splits=5):
    """
    Comprehensive cross-validation for model performance
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    results = {
        'accuracies': [],
        'precisions': [],
        'recalls': [],
        'f1_scores': []
    }
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        
        # Prepare PyTorch datasets
        train_dataset = DDosDataset(X_train_scaled, y_train)
        val_dataset = DDosDataset(X_val_scaled, y_val)
        
        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
        
        # Initialize model
        model = model_class(input_dim=input_dim, num_classes=num_classes)
        
        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        
        # Train model
        trained_model, _ = train_ddos_detector(
            model, train_loader, val_loader, criterion, optimizer, 
            num_epochs=20, patience=5
        )
        
        # Evaluate model
        trained_model.eval()
        all_preds = []
        all_true = []
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        trained_model = trained_model.to(device)
        
        with torch.no_grad():
            for batch_features, batch_labels in val_loader:
                batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
                outputs = trained_model(batch_features)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(batch_labels.cpu().numpy())
        
        # Compute metrics
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_true, all_preds, average='weighted'
        )
        
        results['accuracies'].append(accuracy_score(all_true, all_preds))
        results['precisions'].append(precision)
        results['recalls'].append(recall)
        results['f1_scores'].append(f1)
        
        print(f"\nFold {fold} Results:")
        print(classification_report(all_true, all_preds))
    
    # Print overall cross-validation results
    print("\nCross-Validation Summary:")
    for metric, values in results.items():
        print(f"{metric.capitalize()}: {np.mean(values):.4f} (±{np.std(values):.4f})")
    
    return results

In [39]:
def main():
    # Set random seed for reproducibility
    set_seed(42)
    
    # Load the dataset
    # Note: Replace with your actual path to the dataset
    df = pd.read_csv('/Users/Viku/Datasets/IDS/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
    
    # Preprocess the data with enhanced method
    data_prep = robust_preprocess_ddos_data(df)
    
    # Create datasets and loaders
    train_dataset = DDosDataset(data_prep['X_train'], data_prep['y_train'])
    test_dataset = DDosDataset(data_prep['X_test'], data_prep['y_test'])
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    # Initialize the neural network model
    model = AdvancedDDosDetector(
        input_dim=data_prep['input_dim'], 
        num_classes=data_prep['num_classes']
    )
    
    # Loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=0.001, 
        weight_decay=1e-4
    )
    
    # Train the model
    trained_model, training_history = train_ddos_detector(
        model, 
        train_loader, 
        test_loader, 
        criterion, 
        optimizer,
        num_epochs=30, 
        patience=7
    )
    
    # Perform comprehensive cross-validation
    cv_results = comprehensive_model_validation(
        data_prep['X_train'], 
        data_prep['y_train'], 
        AdvancedDDosDetector, 
        data_prep['input_dim'], 
        data_prep['num_classes']
    )
    
    # Final model evaluation on test set
    trained_model.eval()
    all_preds = []
    all_true = []
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    trained_model = trained_model.to(device)
    
    with torch.no_grad():
        for batch_features, batch_labels in test_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = trained_model(batch_features)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(batch_labels.cpu().numpy())
    
    # Generate detailed classification report
    print("\nFinal Test Set Classification Report:")
    print(classification_report(
        all_true, 
        all_preds, 
        target_names=data_prep['label_encoder'].classes_
    ))
    
    # Confusion Matrix Visualization
    cm = confusion_matrix(all_true, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues', 
        xticklabels=data_prep['label_encoder'].classes_,
        yticklabels=data_prep['label_encoder'].classes_
    )
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.close()


In [40]:
if __name__ == '__main__':
    main()

Original Class Distribution:
Label
DDoS      0.567131
BENIGN    0.432869
Name: proportion, dtype: float64

Final Dataset Info:
Total features: 5
Train samples: 180596
Test samples: 45149

Class Distribution:
Train: [0.43286673 0.56713327]
Test: [0.4328778 0.5671222]
Using device: mps


Training Progress:   3%|▎         | 1/30 [00:21<10:34, 21.87s/it]

Epoch 1: Train Loss: 0.2373, Train Acc: 0.8852, Val Loss: 0.1540, Val Acc: 0.9393


Training Progress:   7%|▋         | 2/30 [00:42<09:53, 21.19s/it]

Epoch 2: Train Loss: 0.1703, Train Acc: 0.9289, Val Loss: 0.1249, Val Acc: 0.9509


...

Epoch 13: Train Loss: 0.1335, Train Acc: 0.9486, Val Loss: 0.1180, Val Acc: 0.9546


Training Progress:  65%|██████▌   | 13/20 [03:52<02:05, 17.91s/it]

Epoch 14: Train Loss: 0.1336, Train Acc: 0.9495, Val Loss: 0.1129, Val Acc: 0.9583
Early stopping triggered



Fold 5 Results:
              precision    recall  f1-score   support

           0       1.00      0.90      0.95     15634
           1       0.93      1.00      0.96     20485

    accuracy                           0.96     36119
   macro avg       0.97      0.95      0.96     36119
weighted avg       0.96      0.96      0.96     36119


Cross-Validation Summary:
Accuracies: 0.9595 (±0.0015)
Precisions: 0.9622 (±0.0013)
Recalls: 0.9595 (±0.0015)
F1_scores: 0.9592 (±0.0016)

Final Test Set Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      0.92      0.96     19544
        DDoS       0.94      1.00      0.97     25605

    accuracy                           0.97     45149
   macro avg       0.97      0.96      0.97     45149
weighted avg       0.97      0.97      0.97     45149



<Figure size 1200x600 with 0 Axes>